# Notebook 02: detección, segmentación y pose sobre video

Corre tres modelos sobre cada fotograma: el detector propio (`mate`, `termo`), la segmentación de COCO sin personas y la pose de las personas. Genera un `.mp4` por cada video de entrada.

Antes de correrlo hace falta:

- haber ejecutado el notebook 01, o tener `models/best.pt`;
- 2 o 3 videos, grabados por el grupo o descargados con permiso, donde aparezcan un mate, un termo, personas y otros objetos de COCO. Si son descargados, anotar la fuente y la licencia.

No hace falta que las tres tareas aparezcan en todos los fotogramas.

## 1. Entorno y rutas

In [ ]:
%pip -q install -U "ultralytics>=8.4" pandas matplotlib


In [ ]:
import os
import shutil
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Video, display
from ultralytics import YOLO

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')


def resolve_root() -> Path:
    """Devuelve la raíz del proyecto según el entorno de ejecución.

    En Colab monta Google Drive y usa ``MyDrive/Proyecto_Integrador_YOLO26``
    si existe; si no, ``/content/CV-mate-detection``. En local usa el
    directorio padre cuando el notebook corre desde ``notebooks/``.

    Returns:
        Directorio raíz del repositorio.
    """
    if not IN_COLAB:
        cwd = Path.cwd()
        return cwd.parent if cwd.name == 'notebooks' else cwd
    colab_repo = Path('/content/CV-mate-detection')
    try:
        from google.colab import drive

        drive.mount('/content/drive')
    except Exception as exc:  # noqa: BLE001 - Drive es opcional.
        print('Advertencia al montar Drive:', exc)
        return colab_repo if colab_repo.exists() else Path.cwd()
    drive_root = Path('/content/drive/MyDrive/Proyecto_Integrador_YOLO26')
    return drive_root if drive_root.exists() else colab_repo


def resolve_best_weights(root: Path) -> Path:
    """Busca los pesos entrenados en el notebook 01.

    Args:
        root: Raíz del repositorio.

    Returns:
        ``entrenamientos/mate_termo_yolo26/weights/best.pt`` si existe; si
        no, ``models/best.pt`` (exista o no).
    """
    run_best = root / 'entrenamientos' / 'mate_termo_yolo26' / 'weights'
    run_best = run_best / 'best.pt'
    return run_best if run_best.exists() else root / 'models' / 'best.pt'


ROOT = resolve_root()
BEST = resolve_best_weights(ROOT)
VIDEOS_IN = ROOT / 'videos' / 'input'
VIDEOS_OUT = ROOT / 'videos' / 'output'
VIDEOS_IN.mkdir(parents=True, exist_ok=True)
VIDEOS_OUT.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

print('=== ENTORNO DE EJECUCIÓN ===')
print('Modo Colab:', IN_COLAB)
print('Directorio raíz:', ROOT)
print('Device:', DEVICE)
print('Pesos custom (BEST):', BEST, '| existe:', BEST.exists())
if not BEST.exists():
    print('[AVISO] Falta best.pt: ejecutar primero el notebook 01.')


## 2. Videos de entrada

Poner los videos (`.mp4`, `.mov`, `.mkv`, `.avi` o `.webm`) en `videos/input/`. El enunciado permite hasta 2 minutos por video y recomienda 2 o 3 de unos 20 segundos. Para videos descargados, por ejemplo de Pexels, anotar la URL, la licencia y la fecha.

Entre todos los videos tiene que haber un mate y un termo, personas para la pose y objetos de COCO para las máscaras (mesa, silla, vaso). Conviene revisarlos antes de procesar.

In [ ]:
VIDEO_EXT = {'.mp4', '.mov', '.mkv', '.avi', '.webm'}
MAX_SECONDS = 120


def video_info(path: Path) -> dict:
    """Lee duración, FPS y resolución de un video.

    Args:
        path: Video a inspeccionar.

    Returns:
        Diccionario con ``video``, ``segundos`` (-1 si el FPS es inválido),
        ``fps`` y ``resolucion``.
    """
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return {
        'video': path.name,
        'segundos': round(frames / fps, 1) if fps > 0 else -1,
        'fps': round(fps, 2),
        'resolucion': f'{width}x{height}',
    }


videos = sorted(p for p in VIDEOS_IN.iterdir()
                if p.is_file() and p.suffix.lower() in VIDEO_EXT)
if not videos:
    raise FileNotFoundError(f'No hay videos de entrada: subir 2-3 a '
                            f'{VIDEOS_IN}.')
info = pd.DataFrame([video_info(p) for p in videos])
display(info)
if ((info.segundos <= 0) | (info.segundos > MAX_SECONDS)).any():
    raise ValueError(f'Hay videos ilegibles o de más de {MAX_SECONDS} s.')


## 3. Modelos y umbrales

- `best.pt` detecta `mate` y `termo` y los dibuja como cajas con su confianza.
- `yolo26n-seg.pt` segmenta las clases de COCO menos `person`. Si una máscara cae en un 50 % o más dentro de una caja de mate o termo, no se dibuja. Pasa, por ejemplo, cuando un termo se detecta como `bottle`.
- `yolo26n-pose.pt` dibuja keypoints y esqueleto, y mantiene un ID por persona entre fotogramas.

Solo entrenamos el detector; la segmentación y la pose usan los pesos preentrenados en COCO.

In [ ]:
if not BEST.exists():
    raise FileNotFoundError(f'Falta best.pt en {BEST}: ejecutar el '
                            'notebook 01.')

# Modelos YOLO26 preentrenados en COCO (Ultralytics >= 8.4 los descarga).
SEG_MODEL_NAME = 'yolo26n-seg.pt'
POSE_MODEL_NAME = 'yolo26n-pose.pt'

DETECTOR = YOLO(str(BEST))
SEGMENTADOR = YOLO(SEG_MODEL_NAME)

CONF_CUSTOM = 0.35
CONF_SEG = 0.45
CONF_POSE = 0.35
CONF_KEYPOINT = 0.35
MASK_ALPHA = 0.35
# Fracción del área de una máscara COCO dentro de una caja mate/termo a
# partir de la cual se descarta (p. ej. "bottle" sobre un termo).
MASK_OVERLAP_SUPPRESS = 0.5
IMGSZ = 640

expected_tasks = {'detector': (DETECTOR, 'detect'),
                  'segmentador': (SEGMENTADOR, 'segment'),
                  'pose': (YOLO(POSE_MODEL_NAME), 'pose')}
for label, (model, task) in expected_tasks.items():
    if model.task != task:
        raise ValueError(f'El modelo {label} es de tipo {model.task}, '
                         f'se esperaba {task}.')

print('Clases del detector personalizado:', DETECTOR.names)
print('Modelos cargados:')
print('  - Detección custom:', BEST.name)
print('  - Segmentación COCO:', SEG_MODEL_NAME)
print('  - Pose:', POSE_MODEL_NAME)


## 4. Funciones de dibujo

Todas las funciones dibujan sobre el fotograma BGR. La clase `person` se filtra por nombre. La regla que oculta máscaras sobre mate y termo es una heurística, así que conviene revisar el resultado a ojo: la confusión entre termo y botella es frecuente.

In [ ]:
# Colores en BGR (convención de OpenCV).
COLORES_CUSTOM = {'mate': (30, 230, 30), 'termo': (40, 160, 255)}
COLORES_SEG = [(205, 130, 40), (180, 60, 180), (40, 170, 170),
               (110, 210, 60), (200, 80, 80)]
COLOR_ESQUELETO = (230, 65, 240)
COLOR_KEYPOINT = (40, 255, 255)
BLANCO = (255, 255, 255)
# Pares de keypoints COCO (17 puntos) que forman el esqueleto.
COCO_SKELETON = [(0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9),
                 (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13),
                 (13, 15), (12, 14), (14, 16)]


def draw_text(img: np.ndarray, text: str, x: float, y: float,
              color: tuple[int, int, int] = BLANCO,
              scale: float = 0.5) -> None:
    """Escribe texto con fondo oscuro, sin salirse del cuadro.

    Args:
        img: Fotograma BGR; se modifica en el lugar.
        text: Texto a escribir.
        x: Coordenada horizontal de la esquina inferior izquierda.
        y: Coordenada vertical de la línea base.
        color: Color del texto en BGR.
        scale: Escala de la fuente.
    """
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), base = cv2.getTextSize(text, font, scale, 1)
    x = max(0, min(int(x), img.shape[1] - tw - 6))
    y = max(th + 6, min(int(y), img.shape[0] - base - 2))
    cv2.rectangle(img, (x, y - th - 5), (x + tw + 5, y + base + 2),
                  (20, 20, 20), -1)
    cv2.putText(img, text, (x + 2, y), font, scale, color, 1, cv2.LINE_AA)


def boxes_to_mask(boxes: list[list[float]],
                  shape: tuple[int, int]) -> np.ndarray:
    """Rasteriza cajas ``xyxy`` en una máscara binaria.

    Args:
        boxes: Cajas ``[x1, y1, x2, y2]`` en píxeles.
        shape: ``(alto, ancho)`` de la máscara.

    Returns:
        Máscara ``uint8`` con 1 dentro de alguna caja.
    """
    height, width = shape
    mask = np.zeros((height, width), dtype=np.uint8)
    for x1, y1, x2, y2 in boxes:
        x1, x2 = (int(np.clip(v, 0, width)) for v in (x1, x2))
        y1, y2 = (int(np.clip(v, 0, height)) for v in (y1, y2))
        mask[y1:y2, x1:x2] = 1
    return mask


def apply_segmentation(img: np.ndarray, result,
                       custom_boxes: list[list[float]]) -> int:
    """Superpone las máscaras COCO, excepto personas y clases propias.

    Se omiten las máscaras de ``person`` y las que caen mayormente dentro de
    una caja del detector propio (``MASK_OVERLAP_SUPPRESS``). Las máscaras
    se mezclan con ``MASK_ALPHA`` y después se dibujan contornos y etiquetas
    nítidos encima.

    Args:
        img: Fotograma BGR; se modifica en el lugar.
        result: ``Results`` de Ultralytics del modelo de segmentación.
        custom_boxes: Cajas ``xyxy`` de mate/termo del mismo fotograma.

    Returns:
        Cantidad de máscaras dibujadas.
    """
    if result.masks is None or result.boxes is None:
        return 0
    height, width = img.shape[:2]
    custom_area = boxes_to_mask(custom_boxes, (height, width))
    overlay = img.copy()
    annotations = []
    masks = result.masks.data.cpu().numpy()
    classes = result.boxes.cls.int().cpu().tolist()
    scores = result.boxes.conf.cpu().numpy()
    for raw_mask, cls_id, score in zip(masks, classes, scores, strict=True):
        name = str(result.names[cls_id]).lower()
        if name == 'person':
            continue
        mask = (raw_mask > 0.5).astype(np.uint8)
        if mask.shape != (height, width):
            mask = cv2.resize(mask, (width, height),
                              interpolation=cv2.INTER_NEAREST)
        area = int(mask.sum())
        if area == 0:
            continue
        if (mask & custom_area).sum() / area >= MASK_OVERLAP_SUPPRESS:
            continue
        color = COLORES_SEG[cls_id % len(COLORES_SEG)]
        overlay[mask.astype(bool)] = color
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)
        annotations.append((contours, color, f'COCO {name} {score:.2f}'))

    img[:] = cv2.addWeighted(overlay, MASK_ALPHA, img, 1 - MASK_ALPHA, 0)
    for contours, color, label in annotations:
        cv2.drawContours(img, contours, -1, color, 1)
        if contours:
            x, y, _, _ = cv2.boundingRect(max(contours, key=cv2.contourArea))
            draw_text(img, label, x, y - 4, color)
    return len(annotations)


def draw_pose(img: np.ndarray, result) -> int:
    """Dibuja keypoints, esqueleto e ID de seguimiento de cada persona.

    Solo se dibujan los keypoints con confianza ``>= CONF_KEYPOINT``.

    Args:
        img: Fotograma BGR; se modifica en el lugar.
        result: ``Results`` de Ultralytics del modelo de pose.

    Returns:
        Cantidad de personas detectadas.
    """
    if result.keypoints is None:
        return 0
    points = result.keypoints.xy.cpu().numpy()
    if result.keypoints.conf is not None:
        confidence = result.keypoints.conf.cpu().numpy()
    else:
        confidence = np.ones(points.shape[:2], dtype=float)
    track_ids = None
    if result.boxes is not None and result.boxes.id is not None:
        track_ids = result.boxes.id.int().cpu().numpy()

    for person, (joints, probs) in enumerate(zip(points, confidence,
                                                 strict=True)):
        for a, b in COCO_SKELETON:
            if max(a, b) >= len(joints):
                continue
            if min(probs[a], probs[b]) < CONF_KEYPOINT:
                continue
            pa = tuple(map(int, joints[a]))
            pb = tuple(map(int, joints[b]))
            if min(*pa, *pb) < 0:
                continue
            cv2.line(img, pa, pb, COLOR_ESQUELETO, 2, cv2.LINE_AA)
        for (x, y), prob in zip(joints, probs, strict=True):
            if prob >= CONF_KEYPOINT and x > 0 and y > 0:
                cv2.circle(img, (int(x), int(y)), 3, COLOR_KEYPOINT, -1,
                           cv2.LINE_AA)
        visible = np.flatnonzero(probs >= CONF_KEYPOINT)
        if len(visible):
            label = 'Persona'
            if track_ids is not None:
                label += f' ID {track_ids[person]}'
            x, y = joints[visible[0]]
            draw_text(img, label, x, y - 10, COLOR_ESQUELETO)
    return len(points)


def draw_custom(img: np.ndarray, result) -> Counter:
    """Dibuja las cajas del detector propio (mate y termo) con su confianza.

    Args:
        img: Fotograma BGR; se modifica en el lugar.
        result: ``Results`` de Ultralytics del detector propio.

    Returns:
        Detecciones por nombre de clase.
    """
    count = Counter()
    if result.boxes is None:
        return count
    for box, cls_id, score in zip(result.boxes.xyxy.cpu().numpy(),
                                  result.boxes.cls.int().cpu().tolist(),
                                  result.boxes.conf.cpu().numpy(),
                                  strict=True):
        name = str(result.names[cls_id]).lower()
        x1, y1, x2, y2 = map(int, box)
        count[name] += 1
        color = COLORES_CUSTOM.get(name, (0, 255, 0))
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        draw_text(img, f'{name} {score:.2f}', x1, y1 - 6, color)
    return count


## 5. Procesamiento

Cada fotograma pasa por los tres modelos y se dibuja en este orden: máscaras, pose y cajas propias, para que las cajas queden siempre visibles. El video de salida mantiene la resolución y los FPS del original. Al final se convierte a H.264 con `ffmpeg` para que Colab lo reproduzca.

Los IDs del tracker siguen a cada persona de un fotograma al siguiente, pero pueden cambiar cuando alguien queda tapado.

In [ ]:
TITULO = 'YOLO26 | deteccion + segmentacion COCO + pose'


def open_video(source: Path) -> tuple[cv2.VideoCapture, float, int, int, int]:
    """Abre un video y valida sus propiedades.

    Args:
        source: Video de entrada.

    Returns:
        ``(captura, fps, cuadros, ancho, alto)``.

    Raises:
        RuntimeError: Si el video es ilegible o dura más de ``MAX_SECONDS``.
    """
    cap = cv2.VideoCapture(str(source))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if min(fps, count, width, height) <= 0:
        cap.release()
        raise RuntimeError(f'Video inválido: {source}')
    if count / fps > MAX_SECONDS:
        cap.release()
        raise RuntimeError(f'Supera {MAX_SECONDS} segundos: {source}')
    return cap, fps, count, width, height


def annotate_frame(frame: np.ndarray,
                   pose_model: YOLO) -> tuple[np.ndarray, Counter, int, int]:
    """Corre los tres modelos sobre un fotograma y compone el resultado.

    Orden de dibujo: máscaras COCO, luego pose y por último las cajas
    propias, para que estas queden siempre visibles.

    Args:
        frame: Fotograma BGR original (no se modifica).
        pose_model: Modelo de pose con el tracker de este video.

    Returns:
        ``(fotograma_anotado, detecciones_por_clase, mascaras, personas)``.
    """
    custom = DETECTOR.predict(frame, conf=CONF_CUSTOM, imgsz=IMGSZ,
                              device=DEVICE, verbose=False)[0]
    coco = SEGMENTADOR.predict(frame, conf=CONF_SEG, imgsz=IMGSZ,
                               device=DEVICE, retina_masks=True,
                               verbose=False)[0]
    human = pose_model.track(frame, conf=CONF_POSE, imgsz=IMGSZ,
                             device=DEVICE, persist=True,
                             tracker='bytetrack.yaml', verbose=False)[0]
    custom_boxes = (custom.boxes.xyxy.cpu().numpy().tolist()
                    if custom.boxes is not None else [])
    painted = frame.copy()
    n_masks = apply_segmentation(painted, coco, custom_boxes)
    n_people = draw_pose(painted, human)
    detections = draw_custom(painted, custom)
    draw_text(painted, TITULO, 10, 24)
    return painted, detections, n_masks, n_people


def transcode_h264(raw: Path, destination: Path) -> None:
    """Convierte el MP4 temporal a H.264/yuv420p para verlo en Colab.

    Si ``ffmpeg`` no está disponible o falla, se conserva el MP4 original.

    Args:
        raw: MP4 escrito por OpenCV (códec ``mp4v``); se elimina al terminar.
        destination: Ruta del video final.
    """
    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw),
           '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '23',
           '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
           str(destination)]
    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except (OSError, subprocess.CalledProcessError) as exc:
        shutil.move(raw, destination)
        print('Advertencia: sin conversión H.264, se conserva el MP4 '
              'temporal:', exc)
    else:
        raw.unlink(missing_ok=True)


def process_video(source: Path, destination: Path) -> dict:
    """Procesa un video completo con detección, segmentación y pose.

    Se crea un modelo de pose nuevo por video para no arrastrar el estado
    del tracker entre videos.

    Args:
        source: Video de entrada.
        destination: Ruta del MP4 de salida.

    Returns:
        Resumen con fotogramas procesados, detecciones por clase, máscaras,
        poses y tiempo de ejecución.

    Raises:
        RuntimeError: Si el video es inválido o no se pudo escribir.
    """
    cap, fps, count, width, height = open_video(source)
    raw = destination.with_name(f'{destination.stem}_temporal.mp4')
    writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*'mp4v'), fps,
                             (width, height))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError('No se pudo crear el video temporal.')

    pose_model = YOLO(POSE_MODEL_NAME)
    detections = Counter()
    n_masks = 0
    n_people = 0
    processed = 0
    log_every = max(1, int(fps * 5))
    start = time.time()
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            painted, found, masks, people = annotate_frame(frame, pose_model)
            detections.update(found)
            n_masks += masks
            n_people += people
            writer.write(painted)
            processed += 1
            if processed % log_every == 0:
                print(f'{source.name}: {processed}/{count} cuadros')
    finally:
        writer.release()
        cap.release()
    if processed == 0:
        raise RuntimeError(f'No se decodificaron fotogramas de {source}.')

    transcode_h264(raw, destination)
    return {
        'video': source.name,
        'salida': destination.name,
        'fotogramas': processed,
        'detecciones_mate': detections['mate'],
        'detecciones_termo': detections['termo'],
        'mascaras_COCO': n_masks,
        'poses_personas': n_people,
        'segundos_ejecucion': round(time.time() - start, 1),
    }


resultados = [process_video(v, VIDEOS_OUT / f'{v.stem}_yolo26.mp4')
              for v in videos]
resumen = pd.DataFrame(resultados)
display(resumen)
resumen.to_csv(VIDEOS_OUT / 'resumen_procesamiento.csv', index=False)


## 6. Revisión visual

La celda muestra un fotograma, tomado a un tercio de cada video, del original y del procesado, y embebe el video procesado si pesa menos de 20 MB. Igual hay que mirar los videos completos.

In [ ]:
MAX_EMBED_BYTES = 20_000_000  # Colab no muestra bien videos más pesados.


def frame_at(path: Path, fraction: float = 0.33) -> np.ndarray | None:
    """Extrae un fotograma en una posición relativa del video.

    Args:
        path: Video a leer.
        fraction: Posición entre 0 (inicio) y 1 (final).

    Returns:
        Fotograma RGB, o ``None`` si no se pudo leer.
    """
    cap = cv2.VideoCapture(str(path))
    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, int(count * fraction)))
    ok, frame = cap.read()
    cap.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ok else None


for row in resultados:
    original = VIDEOS_IN / row['video']
    processed = VIDEOS_OUT / row['salida']
    before, after = frame_at(original), frame_at(processed)
    if before is None or after is None:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(before)
    axes[0].set_title(f'Original: {original.name}')
    axes[1].imshow(after)
    axes[1].set_title(f'Procesado: {processed.name}')
    for ax in axes:
        ax.axis('off')
    fig.tight_layout()
    plt.show()
    print('Video resultante:', processed)
    if processed.stat().st_size < MAX_EMBED_BYTES:
        display(Video(str(processed), embed=True))


## 7. Cobertura y conclusiones

Los totales cuentan apariciones por fotograma, no objetos distintos. Entre todos los videos debería haber detecciones de las dos clases, máscaras de COCO y poses.

Si falta alguna, primero hay que revisar que el objeto se vea en el video. Si se ve y no se detecta, probar con otro video o revisar el detector.

In [ ]:
if not resultados:
    raise RuntimeError('No hay videos procesados.')
COLUMNAS = ['detecciones_mate', 'detecciones_termo', 'mascaras_COCO',
            'poses_personas']
totals = resumen[COLUMNAS].sum()
print('Eventos observados por fotograma:')
print(totals.to_string())
missing = [name for name, value in totals.items() if value == 0]
if missing:
    print('ATENCIÓN: no se observó evidencia visual de:', ', '.join(missing))
    print('Elegir otros videos o revisar parámetros, etiquetas y modelo.')
else:
    print('Hay evidencia de las cuatro salidas esperadas; revisar la calidad '
          'visualmente.')
print('Archivos finales en:', VIDEOS_OUT)


## 8. Entrega

Se entregan los dos notebooks, el dataset con su `data.yaml`, `best.pt`, el origen y la licencia de fotos y videos, los videos originales y procesados, y las conclusiones. Las conclusiones explican qué detecta bien, los falsos positivos, las limitaciones, los umbrales usados, los fotogramas por segundo procesados y qué se podría mejorar.

Que exista el MP4 no alcanza para dar el resultado por bueno. Hay que revisar las máscaras, el esqueleto, los nombres de clase y la confianza en fotogramas reales.